In [1]:



!pip install pydicom numpy matplotlib scikit-image scipy trimesh

In [2]:
import os
import pydicom
import numpy as np
import matplotlib.pyplot as plt

from collections import defaultdict
from skimage import measure
from scipy import ndimage
import trimesh
print("All libraries installed successfully")

All libraries installed successfully


In [3]:
root_path = r"C:\Sanvi\hipjoint\manifest-1773177349818"

print("Scanning dataset for CT series...")


Scanning dataset for CT series...


In [4]:
series_dict = defaultdict(list)

for root, dirs, files in os.walk(root_path):

    for file in files:

        path = os.path.join(root, file)

        try:
            ds = pydicom.dcmread(path, stop_before_pixels=True)

            series_uid = ds.SeriesInstanceUID

            series_dict[series_uid].append(path)

        except:
            continue


print("Total CT series found:", len(series_dict))
output_folder = "stl_models"

os.makedirs(output_folder, exist_ok=True)
for idx, (series_id, file_list) in enumerate(series_dict.items()):

    print("\n=================================")
    print("Processing series", idx+1)
    print("Series UID:", series_id)
    print("Slices found:", len(file_list))


    slices = [pydicom.dcmread(f) for f in file_list]

    try:
        slices.sort(key=lambda x: float(x.ImagePositionPatient[2]))
    except:
        print("Skipping series due to missing slice position")
        continue


    volume = np.stack([s.pixel_array for s in slices])

    print("Volume shape:", volume.shape)
    volume = volume.astype(np.int16)

    for i in range(len(slices)):

        intercept = slices[i].RescaleIntercept
        slope = slices[i].RescaleSlope

        volume[i] = slope * volume[i] + intercept


    print("HU conversion done")
    bone_mask = (volume > 150) & (volume < 2000)

    bone_mask = ndimage.binary_closing(bone_mask, iterations=2)
    bone_mask = ndimage.binary_opening(bone_mask, iterations=2)


    print("Bone segmentation completed")
    coords = np.argwhere(bone_mask)

    if len(coords) == 0:
        print("No bone detected, skipping")
        continue

    min_z, min_y, min_x = coords.min(axis=0)
    max_z, max_y, max_x = coords.max(axis=0)

    cropped_mask = bone_mask[min_z:max_z, min_y:max_y, min_x:max_x]

    print("Cropped volume shape:", cropped_mask.shape)
    print("Running marching cubes...")

    verts, faces, normals, values = measure.marching_cubes(cropped_mask, level=0)

    print("Vertices:", len(verts))
    print("Faces:", len(faces))
    mesh = trimesh.Trimesh(vertices=verts, faces=faces)
    print("Mesh created")
    print("Repairing mesh...")
    mesh.process()
    trimesh.repair.fix_normals(mesh)
    trimesh.repair.fill_holes(mesh)
    print("Mesh repair done")
    print("Applying mesh smoothing...")
    trimesh.smoothing.filter_laplacian(mesh)
    print("Mesh smoothing applied")
    print("Mesh simplified")
    output_path = os.path.join(output_folder, f"hip_model_{idx+1}.stl")

    mesh.export(output_path)

    print("STL exported:", output_path)
    break

print("\nPipeline finished successfully")


Total CT series found: 116

Processing series 1
Series UID: 1.3.6.1.4.1.14519.5.2.1.7085.2036.915272436289218804778532165447
Slices found: 153
Volume shape: (153, 512, 512)
HU conversion done
Bone segmentation completed
Cropped volume shape: (148, 319, 413)
Running marching cubes...
Vertices: 288616
Faces: 578536
Mesh created
Repairing mesh...
Mesh repair done
Applying mesh smoothing...


C:\Users\DELL\AppData\Local\Programs\Python\Python313\Lib\site-packages\trimesh\smoothing.py:92: RuntimeWarning: invalid value encountered in scalar power
  vertices *= (vol_ini / vol_new) ** (1.0 / 3.0)


Mesh smoothing applied
Mesh simplified
STL exported: stl_models\hip_model_1.stl

Pipeline finished successfully
